In [1]:
import json
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import time

from rich.progress import Progress

from python_magnetrun.MagnetRun import MagnetRun, load_mrun
from python_magnetrun.signature import Signature

In [2]:
DATA_DIR = Path("../Data")
PUPITRE_ROOT = DATA_DIR / "pupitre_2023" / "srv-data-install"
PUPITRE_DIR = PUPITRE_ROOT / "M9"
PIGBROTHER = DATA_DIR / "pigbrother_2025" / "M10_Overview_251201-0909.tdms"
DB = Path("../to_duckdb/magnetdb.duckdb")

FIELD_THRESHOLD = 0.1

# HOUSING SUMMARY
### Load and merge the housing summary files


In [4]:
PATH_COLUMNS = [
    "overview", "archive", "pupitre", "default", "trigger", "spike",
    "hybrid_kHz", "hybrid_rms", "hybrid_trigger", "hybrid_vprocess",
    "pigbrother_runlog", "pupitre_runlog",
]

rows = []
for file in sorted(DATA_DIR.glob("*_summary-*.json")):

    print(f"Loading {file.name}")

    housing = file.stem.split("_")[0]
    year = int(file.stem[-4: ])

    with open(file, "r") as f:
        data = json.load(f)

    df = pd.json_normalize(data)

    for col in PATH_COLUMNS:
        df[col] = df[col].apply(lambda x: Path(x).name if x else x)

    df["housing"] = housing
    df["year"] = year

    rows.append(df)

summary_df = pd.concat(rows, ignore_index = True)

summary_df["experiment_id"]       = None
summary_df["field_max"]           = pd.Series(dtype = "float64")
summary_df["field_mean"]          = pd.Series(dtype = "float64")
summary_df["field_time_on"]       = pd.Series(dtype = "float64")
summary_df["mode"]                = ""
summary_df["field_signature"]     = ""
summary_df["reference_signature"] = ""

print(f"Found {len(summary_df)} summary files")
print(summary_df.head())

Loading M10_summary-2022.json
Loading M10_summary-2023.json
Loading M10_summary-2024.json
Loading M10_summary-2025.json
Loading M10_summary-2026.json
Loading M9_summary-2023.json
Loading M9_summary-2024.json
Loading M9_summary-2025.json
Loading M9_summary-2026.json
Found 1524 summary files
                   filename                       overview  \
0  M10_Overview_220204-1003  M10_Overview_220204-1003.tdms   
1  M10_Overview_220204-1510  M10_Overview_220204-1510.tdms   
2  M10_Overview_220206-1521  M10_Overview_220206-1521.tdms   
3  M10_Overview_220208-0951  M10_Overview_220208-0951.tdms   
4  M10_Overview_220211-0941  M10_Overview_220211-0941.tdms   

                        archive                    pupitre default trigger  \
0  M10_Archive_220204-1003.tdms  2022.02.04 - 10:04:03.txt                   
1  M10_Archive_220204-1510.tdms  2022.02.04 - 15:10:08.txt                   
2  M10_Archive_220206-1521.tdms  2022.02.06 - 15:21:53.txt                   
3  M10_Archive_220208-09

### Refresh the housing summary table and display basic stats

In [5]:
con = duckdb.connect(DB)
con.execute(
    """
        DROP TABLE IF EXISTS housing_summary
    """
)
con.register("summary_df", summary_df)
con.execute(
    """
        CREATE TABLE housing_summary AS
        SELECT *
        FROM summary_df
    """
)

print("\nRows:\n",
    con.execute(
        """
            SELECT housing, year, COUNT(*) AS n
            FROM housing_summary
            GROUP BY housing, year
            ORDER BY housing, year
        """
    ).fetchdf()
)


Rows:
   housing  year    n
0     M10  2022  238
1     M10  2023  209
2     M10  2024  146
3     M10  2025  233
4     M10  2026   66
5      M9  2023  222
6      M9  2024  162
7      M9  2025  191
8      M9  2026   57


### Data quality audit

In [6]:
## Check the date schema
print("\nTABLE SCHEMA: ",
    con.execute(
        """
            DESCRIBE housing_summary
        """
    ).fetchdf()
)
## Count imported records
print("\nNUMBER OF ROWS:", 
    con.execute(
        """
            SELECT COUNT(*) FROM housing_summary
        """
    ).fetchone()[0]
)
# Count number of records linked to experiments
print("NUMBER OF MATCHES:",
    con.execute(
        """
            SELECT COUNT(*)
            FROM housing_summary AS h
            JOIN experiments AS e
            ON h.pupitre LIKE '%' || e.file
        """).fetchone()[0]
)


TABLE SCHEMA:              column_name column_type null   key default extra
0              filename     VARCHAR  YES  None    None  None
1              overview     VARCHAR  YES  None    None  None
2               archive     VARCHAR  YES  None    None  None
3               pupitre     VARCHAR  YES  None    None  None
4               default     VARCHAR  YES  None    None  None
5               trigger     VARCHAR  YES  None    None  None
6                 spike     VARCHAR  YES  None    None  None
7            hybrid_kHz     VARCHAR  YES  None    None  None
8            hybrid_rms     VARCHAR  YES  None    None  None
9        hybrid_trigger     VARCHAR  YES  None    None  None
10      hybrid_vprocess     VARCHAR  YES  None    None  None
11    pigbrother_runlog     VARCHAR  YES  None    None  None
12       pupitre_runlog     VARCHAR  YES  None    None  None
13              housing     VARCHAR  YES  None    None  None
14                 year      BIGINT  YES  None    None  None
15      

In [13]:
# Check for missing files
print("\nMISSING FILES:\n",
    con.execute(
        """
            SELECT
                SUM(CASE WHEN overview = '' THEN 1 ELSE 0 END) AS overview,
                SUM(CASE WHEN archive  = '' THEN 1 ELSE 0 END) AS archive,
                SUM(CASE WHEN pupitre  = '' THEN 1 ELSE 0 END) AS pupiter
            from housing_summary
        """
    ).fetchdf()
)
# Check for duplicate files
print("\nDUPLICATE FILENAMES:\n",
    con.execute(
        """
            SELECT filename, COUNT(*) AS n
            FROM housing_summary
            GROUP BY filename
            HAVING COUNT(*) > 1
            ORDER BY n DESC
        """
    ).fetchdf()
)

# List records for which no Pupitre file is available
print(
    con.execute(
        """
            SELECT filename, housing, year, pupitre
            FROM housing_summary
            WHERE pupitre IS NULL OR pupitre = ''
            ORDER BY year, housing, filename
        """
    ).fetchdf()
)

ConnectionException: Connection Error: Connection already closed!

### Link with the user DB : Add foreign key column and populate

In [8]:
con.execute(
    """
        UPDATE housing_summary AS h
        SET experiment_id = e.id
        FROM experiments AS e
        WHERE h.pupitre LIKE '%' || e.file
    """
)

print("\nLINKED EXPERIMENTS:",
    con.execute(
        """
            SELECT COUNT(*)
            FROM housing_summary
            WHERE experiment_id IS NOT NULL
        """
    ).fetchone()[0]
)
print(
    con.execute(
        """
            SELECT experiment_id, filename, pupitre
            FROM housing_summary
            WHERE experiment_id IS NOT NULL
            LIMIT 10
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT h.experiment_id, e.name, e.file, h.pupitre
            FROM housing_summary AS h
            JOIN experiments AS e
            ON h.experiment_id = e.id
            LIMIT 10
        """
    ).fetchdf()
)

rows = con.execute(
    """
        SELECT rowid, housing, pupitre
        FROM housing_summary
        WHERE pupitre <> ''
    """
).fetchall()


LINKED EXPERIMENTS: 371
   experiment_id                  filename                    pupitre
0              1  M10_Overview_250313-1523  2025.03.13 - 15:14:35.txt
1              2  M10_Overview_250313-1525  2025.03.13 - 15:31:07.txt
2              4  M10_Overview_250313-1539  2025.03.13 - 15:44:41.txt
3              5  M10_Overview_250315-1509  2025.03.15 - 15:09:06.txt
4              9  M10_Overview_250315-1524  2025.03.15 - 16:53:01.txt
5             10  M10_Overview_250316-1331  2025.03.16 - 13:31:37.txt
6             11  M10_Overview_250316-2059  2025.03.16 - 20:59:53.txt
7             12  M10_Overview_250317-0936  2025.03.17 - 09:36:55.txt
8             13  M10_Overview_250408-1736  2025.04.08 - 17:36:07.txt
9             15  M10_Overview_250410-1429  2025.04.10 - 14:29:22.txt
   experiment_id                   name                       file  \
0              1  2025.03.13 - 15:14:35  2025.03.13 - 15:14:35.txt   
1              2  2025.03.13 - 15:31:07  2025.03.13 - 15:31:07.tx

In [9]:
print(
    con.execute("""
        DESCRIBE housing_summary
    """).fetchdf()
)

            column_name column_type null   key default extra
0              filename     VARCHAR  YES  None    None  None
1              overview     VARCHAR  YES  None    None  None
2               archive     VARCHAR  YES  None    None  None
3               pupitre     VARCHAR  YES  None    None  None
4               default     VARCHAR  YES  None    None  None
5               trigger     VARCHAR  YES  None    None  None
6                 spike     VARCHAR  YES  None    None  None
7            hybrid_kHz     VARCHAR  YES  None    None  None
8            hybrid_rms     VARCHAR  YES  None    None  None
9        hybrid_trigger     VARCHAR  YES  None    None  None
10      hybrid_vprocess     VARCHAR  YES  None    None  None
11    pigbrother_runlog     VARCHAR  YES  None    None  None
12       pupitre_runlog     VARCHAR  YES  None    None  None
13              housing     VARCHAR  YES  None    None  None
14                 year      BIGINT  YES  None    None  None
15        experiment_id 

In [ ]:
start = time.perf_counter()

with Progress() as progress:

    task = progress.add_task("Updating field stats", total=len(rows))
    
    for rowid, housing, pupitre in rows:
        filename = Path(pupitre).name
        filepath = PUPITRE_ROOT / housing / filename
        progress.update(task, description=f"{housing}/{filename}")

        if not filepath.exists():
            progress.advance(task)
            continue

        try:
            md = load_mrun(str(filepath), housing = housing).getMData()
            df = md.Data

            field = df["Field"]
            signature = Signature.from_mdata(mdata, "Field", "t", FIELD_THRESHOLD)
            field_signature = json.dumps()

            con.execute(
                """
                    UPDATE housing_summary
                    SET 
                        field_max = ?,
                        field_mean = ?,
                        field_time_on = ?,
                        field_signature = ?
                    WHERE rowid = ?
                """, 
                (float(field.max()), float(field.mean()), int((field > FIELD_THRESHOLD).sum()), field_signature, int(rowid))
            )

        except Exception as e:
            progress.console.print(f"[red]{filename}: {e}[/red]")

        progress.advance(task)

end = time.perf_counter()
print(f"Dataframe updated in {int((end - start) // 60)} m {((end - start) % 60):.2f} s")

Output()

2023.02.01 - 10:06:11.txt: name 'compute_field_signature' is not defined

2023.02.01 - 10:06:11.txt: name 'compute_field_signature' is not defined

2023.02.01 - 10:06:11.txt: name 'compute_field_signature' is not defined

2023.02.01 - 11:22:10.txt: name 'compute_field_signature' is not defined

...

2025.07.12 - 18:57:46.txt: name 'compute_field_signature' is not defined

2025.07.12 - 20:14:13.txt: name 'compute_field_signature' is not defined

2025.07.13 - 14:15:41.txt: name 'compute_field_signature' is not defined

2025.07.13 - 23:20:28.txt: name 'compute_field_signature' is not defined

Dataframe updated in 1 m 34.79 s


In [11]:
# Validate update
print(
    con.execute(
        """
            SELECT COUNT(field_max) AS field_stats, COUNT(field_signature) AS signatures
            FROM housing_summary
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT experiment_id, field_max, field_signature
            FROM housing_summary
            WHERE field_signature <> ''
            LIMIT 10
        """
    ).fetchdf()
)

print(
    con.execute(
        """
            SELECT experiment_id, field_signature
            FROM housing_summary
            WHERE field_signature IS NOT NULL
            LIMIT 5
        """
    ).fetchdf()
)

   field_stats  signatures
0            0        1524
Empty DataFrame
Columns: [experiment_id, field_max, field_signature]
Index: []
   experiment_id field_signature
0           <NA>                
1           <NA>                
2           <NA>                
3           <NA>                
4           <NA>                


In [25]:
for rowid, housing, pupitre in rows:
    filepath = PUPITRE_ROOT / housing / Path(pupitre).name

    if filepath.exists():
        md = load_mrun(str(filepath), housing=housing)
        mdata = md.getMData()

        print("Testing:", filepath)

        signature = Signature.from_mdata(
            mdata,
            "Field",
            "t",
            FIELD_THRESHOLD
        )

        print("Regimes:", len(signature.regimes))
        print("Times:", signature.times)
        print("Values:", signature.values)

        break

Testing: ../Data/pupitre_2023/srv-data-install/M10/2023.02.01 - 10:06:11.txt
Regimes: 417
Times: [0.0, 70.0, 71.0, 73.0, 74.0, 76.0, 77.0, 79.0, 80.0, 82.0, 83.0, 857.0, 858.0, 860.0, 861.0, 863.0, 864.0, 866.0, 867.0, 869.0, 870.0, 872.0, 873.0, 875.0, 876.0, 1058.0, 1059.0, 1061.0, 1062.0, 1064.0, 1065.0, 1067.0, 1068.0, 1070.0, 1071.0, 1073.0, 1074.0, 1121.0, 1122.0, 1124.0, 1125.0, 1127.0, 1128.0, 1130.0, 1131.0, 1133.0, 1134.0, 1136.0, 1137.0, 1184.0, 1185.0, 1187.0, 1188.0, 1190.0, 1191.0, 1193.0, 1194.0, 1196.0, 1197.0, 1199.0, 1200.0, 1202.0, 1203.0, 1253.0, 1254.0, 1256.0, 1257.0, 1259.0, 1260.0, 1262.0, 1263.0, 1265.0, 1266.0, 1268.0, 1269.0, 1316.0, 1317.0, 1319.0, 1320.0, 1322.0, 1323.0, 1325.0, 1326.0, 1328.0, 1329.0, 1331.0, 1332.0, 1412.0, 1413.0, 1415.0, 1416.0, 1418.0, 1419.0, 1421.0, 1422.0, 1424.0, 1425.0, 1427.0, 1428.0, 1487.0, 1488.0, 1490.0, 1491.0, 1493.0, 1494.0, 1496.0, 1497.0, 1499.0, 1500.0, 1502.0, 1503.0, 1550.0, 1551.0, 1553.0, 1554.0, 1556.0, 1557.0, 155

# MODE INFERRING

In [ ]:
mrun = MagnetRun.fromtdms(housing = "M10", filename = str(PIGBROTHER))
mdata = mrun.getMData()
print(mdata)

print(mdata.Data["Courants_Alimentations"].columns)

magnetdata.fromtdms: ../pigbrother_2025/M10_Overview_251201-0909.tdms
magnetrun.fromtdms: start_time=2025-12-01 08:09:13.922483, type=<class 'datetime.datetime'>
MagnetData(Type=1, Groups={'Courants_Alimentations': {'Courant_A1': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A1', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A2': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A2', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A3': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A3', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A4': OrderedDict({'wf_start_time': np.datet

# PROPOSALS

In [ ]:
# Load proposals metadata and parse experiment date ranges

proposals_df = pd.read_csv(DATA_DIR / "proposals.csv")
proposals_df["Debut"] = pd.to_datetime(proposals_df["Debut"], errors = "coerce")
proposals_df["Fin"]   = pd.to_datetime(proposals_df["Fin"],   errors = "coerce")

In [ ]:
# Connect to the database and recreate the proposals table

con = duckdb.connect(DB)

con.execute(
    """
        DROP TABLE IF EXISTS proposals
    """
)
con.register("proposals_df", proposals_df)
con.execute(
    """
        CREATE TABLE proposals AS
        SELECT * FROM proposals_df
    """
)

In [ ]:
# Inspect imported proposal schema as well as the experiments table
 
print(
    con.execute(
        """
            DESCRIBE proposals
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT * FROM proposals 
            LIMIT 10
        """
    ).fetchdf()
)

print(
    con.execute(
        """
            DESCRIBE experiments
        """
    )
)
print(
    con.execute(
        """
            SELECT * FROM experiments
            LIMIT 10
        """
    ).fetchdf()
)


        column_name   column_type null   key default extra
0           Acronym       VARCHAR  YES  None    None  None
1         ProjectID        BIGINT  YES  None    None  None
2      ResearchArea       VARCHAR  YES  None    None  None
3          Facility       VARCHAR  YES  None    None  None
4      ProposalType       VARCHAR  YES  None    None  None
5        accessMode        DOUBLE  YES  None    None  None
6        CallNumber        BIGINT  YES  None    None  None
7                id        BIGINT  YES  None    None  None
8   ExperimentState       VARCHAR  YES  None    None  None
9              Site       VARCHAR  YES  None    None  None
10    ShotsHourDone        DOUBLE  YES  None    None  None
11       EnergyUsed        DOUBLE  YES  None    None  None
12            Debut  TIMESTAMP_NS  YES  None    None  None
13              Fin  TIMESTAMP_NS  YES  None    None  None
       Acronym  ProjectID ResearchArea  Facility ProposalType  accessMode  \
0    GMS06-217       3218           MS

In [ ]:
# Check temporal coverage of the proposal metadata

print(
    con.execute(
        """
            SELECT MIN(file), MAX(file), COUNT(*)
            FROM experiments
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT DISTINCT year
            FROM housing_summary
            ORDER BY year
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT MIN(Debut), MAX(Fin), COUNT(*)
            FROM proposals
        """
    ).fetchdf()
)

                   min(file)                  max(file)  count_star()
0  2025.03.13 - 15:14:35.txt  2026.04.27 - 13:27:14.txt           755
   year
0  2022
1  2023
2  2024
3  2025
4  2026
  min(Debut)   max(Fin)  count_star()
0 2009-01-19 2023-10-27          1712


In [ ]:
# Add proposal column to housing_summary unless it already exists

con.execute(
    """
        ALTER TABLE housing_summary
        ADD COLUMN IF NOT EXISTS proposal VARCHAR;
    """
)

# Link housing records to proposals by magnet site and experiment date

con.execute(
    """
        UPDATE housing_summary AS h
        SET proposal = p.Acronym
        FROM proposals AS p
        WHERE h.pupitre <> '' AND h.pupitre IS NOT NULL
            AND h.site = regexp_replace(p.Site, '[ie]$', '')
            AND strptime(right(replace(h.pupitre, '.txt', ''), 19), '%y.%m.%d - %H:%M:%S')
        BETWEEN CAST(p.Debut AS TIMESTAMP) AND CAST(p.Fin AS TIMESTAMP);
    """
)

In [ ]:
# Validate propsal linkage

print(
    con.execute(
        """
            SELECT COUNT(*) AS total, COUNT(proposal) AS linked
            FROM housing_summary;
        """
    ).fetchdf()
)

   total  linked
0   1524     463


In [ ]:
con.close()

In [ ]:
proposals_df = pd.read_csv(DATA_DIR / "proposals_2026-07-22.csv")
proposals_df["Experiment Start Date"] = pd.to_datetime(proposals_df["Experiment Start Date"], errors = "coerce")
proposals_df["Experiment End Date"]   = pd.to_datetime(proposals_df["Experiment End Date"], errors = "coerce")

print(proposals_df[["Acronym", "Magnet Sites", "Experiment Start Date", "Experiment End Date"]].head(), proposals_df.shape)

     Acronym  Magnet Sites Experiment Start Date Experiment End Date
0  GIS01-226           NaN            2026-10-20          2026-10-25
1        NaN           NaN                   NaT                 NaT
2        NaN           NaN                   NaT                 NaT
3  GIS02-126           NaN                   NaT                 NaT
4        NaN           NaN                   NaT                 NaT (1912, 13)


In [ ]:
# Check Magent Sites in new proposals_2026-07-26.csv

print(proposals_df["Magnet Sites"].dtype)
print(len(proposals_df))
print(proposals_df["Magnet Sites"].notna().sum())

float64
1912
0


In [ ]:
###
print(
    con.execute("""
        SELECT year, COUNT(*)
        FROM housing_summary
        GROUP BY year
        ORDER BY year
    """).fetchdf()
)

   year  count_star()
0  2024           308
1  2025           424
2  2026           123
